# 06 – Ethics, Fairness & Bias (W12)
W12 slide 49–54: đánh giá theo nhóm, datasheet, quyền riêng tư, giới hạn và khai báo AI. Đây là audit hồi cứu của hai winner đã khóa tại W11/W10, không phải đợt tối ưu mới. Các kết quả chỉ được công bố ở cấp tổng hợp. Không suy ra quốc tịch, giới tính, tuổi hoặc các nhóm bảo vệ từ Country. Không kết luận mô hình công bằng chỉ vì gap nhỏ. File dữ liệu gốc và bảng cấp khách không được commit.

In [1]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src'/'fairness_evaluation.py').is_file()), None)
if ROOT is None: raise FileNotFoundError('Mở notebook trong repository.')
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src.pipeline import load_raw_data, clean_transactions, integrate_external_sources
from src.data_quality import missing_id_comparison
from src import classification_evaluation as ce, regression_evaluation as re
from src.fairness_evaluation import (market_groups, classification_by_group, regression_by_group,
    country_selection, privacy_summary, signed_gap, update_card)
REPORTS = ROOT/'reports'; REPORTS.mkdir(exist_ok=True)
raw = load_raw_data(ROOT/'data/raw')
observed_until = raw.InvoiceDate.max()
summary, _, country = missing_id_comparison(raw)
country = country_selection(country)
display(summary.round(4)); display(country.round(4))
summary.to_csv(REPORTS/'w12_missing_id.csv'); country.to_csv(REPORTS/'w12_country_selection.csv')
print('Các quốc gia có >=10 dòng mua hợp lệ nhưng không có dòng biết ID:', ', '.join(country.index[(country.total>=10)&(country.Known_ID==0)]))
clean, _ = clean_transactions(raw)
del raw
transactions = integrate_external_sources(clean)
del clean
snapshots = ce.temporal_datasets(transactions, observed_until, include_spend_target=True)
del transactions


Nguồn: online_retail_II.csv | 1,067,371 dòng


,all_lines,purchase_lines,gross_value_gbp,median_line_gbp,median_quantity,median_unit_price_gbp,all_lines_share,purchase_lines_share,gross_value_gbp_share
id_status,,,,,,,,,
Known_ID,797885,776844,1.708113e+07,12.48,6.0,1.95,0.7724,0.774,0.8689
Missing_ID,235151,226857,2.577915e+06,5.02,1.0,3.29,0.2276,0.226,0.1311


id_status,Known_ID,Missing_ID,total,missing_share,known_line_share
Country,,,,,
United Kingdom,699684.0,224036.0,923720,0.2425,0.9007
EIRE,15529.0,1583.0,17112,0.0925,0.0200
Germany,15788.0,0.0,15788,0.0000,0.0203
France,13036.0,128.0,13164,0.0097,0.0168
Netherlands,4981.0,0.0,4981,0.0000,0.0064
Spain,3560.0,0.0,3560,0.0000,0.0046
Switzerland,2951.0,117.0,3068,0.0381,0.0038
Belgium,2911.0,0.0,2911,0.0000,0.0037
Portugal,2295.0,114.0,2409,0.0473,0.0030


Các quốc gia có >=10 dòng mua hợp lệ nhưng không có dòng biết ID: Hong Kong, Bermuda
[Merge 1 - Country Metadata] Trước: 794,437 | Sau: 794,437 | Tỷ lệ khớp: 99.71%
  -> Các giá trị chưa khớp (2,273 dòng): ['Channel Islands', 'Unspecified', 'West Indies', 'European Community']
[Merge 2 - Season Calendar]  Trước: 794,437 | Sau: 794,437 | Tỷ lệ khớp: 100.00%


## 1. Độ đại diện và quyền riêng tư
Đếm dòng mua hợp lệ sau chuẩn hóa/loại duplicate và trước bỏ ID; không suy ra số khách thiếu định danh. `market` ở mỗi snapshot là UK nếu region lịch sử = Domestic_UK, Unknown nếu chưa rõ, còn lại International. W11 lấy quốc gia phổ biến nhất trước cutoff (không phải phân nhóm Mixed của W9). Chỉ so sánh nội bộ nhà bán lẻ; không có mẫu dân số đối chứng. Các lớp k-anonymity tính theo số khách phân biệt trong test, không dùng ID làm quasi-ID. Generalize các band cố định, suppression k<10 chỉ dành cho minh họa riêng tư, không xóa khách khỏi model test. Không xuất lớp tương đương hay dòng cấp khách. k không đảm bảo không thể tái định danh bằng nguồn ngoài.

In [2]:
representation = pd.DataFrame({role:market_groups(data).value_counts().reindex(['UK','International','Unknown'],fill_value=0) for role,data in snapshots.items()})
display(representation); display(representation.div(representation.sum(axis=0)).round(4))
representation.to_csv(REPORTS/'w12_representation.csv')
privacy = privacy_summary(snapshots['test'], k_target=10)
display(privacy); privacy.to_csv(REPORTS/'w12_privacy.csv')


,train,validation,test
market,,,
UK,4149,4503,4773
International,360,400,445
Unknown,4,6,7


,train,validation,test
market,,,
UK,0.9193,0.9173,0.9135
International,0.0798,0.0815,0.0852
Unknown,0.0009,0.0012,0.0013


,customers,suppressed,k_min,singleton_customers,customers_below_k,k_target
scenario,,,,,,
exact,5225,0,1,5225,5225,10
generalized,5225,0,1,11,172,10
generalized_suppressed,5053,172,10,0,0,10


## 2. Classification theo thị trường
Cố định Random Forest thắng CV của W11 (không chạy lựa chọn mới). Tái tạo đúng threshold F1 từ validation, dùng số đầy đủ thay vì số làm tròn 0,4805 trong báo cáo. Không sử dụng nhãn test cho fit hay threshold. Primary ROC-AUC theo nhóm; ưu tiên gap recall/TPR để giảm bỏ sót cơ hội mua, kiểm tra FPR để hiểu chi phí liên hệ nhầm. Các tỷ lệ và AUC là mô tả: n<30 không công bố metric; tỷ lệ cần denominator≥10, AUC/AP cần ít nhất 10 mỗi lớp. Đây là ngưỡng thực dụng định trước, không chứng minh độ tin cậy hoặc k-anonymity. Nếu N/A, không thay bằng 0. Gap International − UK; không chọn threshold riêng theo nhóm, không áp đặt ngưỡng fairness tuỳ tiện.

In [3]:
X_train, y_train = ce.split_xy(snapshots['train'])
X_val, y_val = ce.split_xy(snapshots['validation'])
clf = ce.make_pipeline(ce.candidate_models()['Random Forest']).fit(X_train, y_train)
threshold, val_f1 = ce.choose_threshold(y_val, clf.predict_proba(X_val)[:,1])
X_test, y_test = ce.split_xy(snapshots['test'])
prob = clf.predict_proba(X_test)[:,1]
groups = market_groups(snapshots['test'])
clf_table = classification_by_group(y_test, prob, groups, threshold)
print(f'Threshold từ validation (full precision): {threshold:.12g}; F1 validation {val_f1:.4f}')
display(clf_table.round(4))
for col in ['recall','fpr','accuracy','roc_auc']:
    print(f'Intl − UK {col}: {signed_gap(clf_table,col):+.4f}')
clf_table.to_csv(REPORTS/'w12_classification_groups.csv')


Threshold từ validation (full precision): 0.480468678468; F1 validation 0.6602


,n,share,positive_support,negative_support,prevalence,selection_rate,accuracy,recall,fpr,precision,f1,roc_auc,average_precision,status
market,,,,,,,,,,,,,,
Overall,5225,1.0000,2236.0,2989.0,0.4279,0.3610,0.7301,0.6064,0.1773,0.7190,0.6579,0.7936,0.7639,descriptive
UK,4773,0.9135,2026.0,2747.0,0.4245,0.3537,0.7301,0.5987,0.1729,0.7186,0.6532,0.7927,0.7606,descriptive
International,445,0.0852,210.0,235.0,0.4719,0.4360,0.7348,0.6810,0.2170,0.7371,0.7079,0.8047,0.8025,descriptive
Unknown,7,0.0013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,n<30: suppressed


Intl − UK recall: +0.0822
Intl − UK fpr: +0.0441
Intl − UK accuracy: +0.0047
Intl − UK roc_auc: +0.0120


## 3. Regression theo thị trường
Cố định Dummy median thắng CV W10; tái fit trên train. Không chọn lại Ridge/OLS dựa vào test. MAE là chỉ tiêu chính; mean error = actual − predicted (dương = dự báo thiếu), RMSE nhạy với đơn lớn, R² không xác định nếu target hằng. Báo cáo mean actual và zero share để đặt chênh lệch vào bối cảnh. Không loại ngoại lai. MAE khác nhau có thể do phân phối chi tiêu khác, không tự chứng minh bất công. Winner dự báo 0 GBP ở lần nghiệm thu W10, không phù hợp để dự báo doanh thu hoặc phân khúc.

In [4]:
X_reg_train, y_reg_train = re.split_xy(snapshots['train'])
reg = re.make_pipeline(re.candidate_models()['Dummy median']).fit(X_reg_train, y_reg_train)
X_reg_test, y_reg_test = re.split_xy(snapshots['test'])
pred = re.predict_gbp(reg, X_reg_test)
assert np.array_equal(y_test, (y_reg_test>0).astype(int))
reg_table = regression_by_group(y_reg_test, pred, groups)
display(reg_table.round(4))
for col in ['mae_gbp','rmse_gbp','mean_error_gbp']:
    print(f'Intl − UK {col}: {signed_gap(reg_table,col):+.2f} GBP')
reg_table.to_csv(REPORTS/'w12_regression_groups.csv')


,n,share,mae_gbp,rmse_gbp,r2,mean_error_gbp,mean_actual_gbp,mean_predicted_gbp,zero_share,status
market,,,,,,,,,,
Overall,5225,1.0000,515.3416,3144.9414,-0.0276,515.3416,515.3416,0.0,0.5721,descriptive
UK,4773,0.9135,467.8408,2787.4855,-0.0290,467.8408,467.8408,0.0,0.5755,descriptive
International,445,0.0852,1032.9346,5726.3650,-0.0336,1032.9346,1032.9346,0.0,0.5281,descriptive
Unknown,7,0.0013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,n<30: suppressed


Intl − UK mae_gbp: +565.09 GBP
Intl − UK rmse_gbp: +2938.88 GBP
Intl − UK mean_error_gbp: +565.09 GBP


## 4. Trình bày kết quả và cập nhật tài liệu
Chỉ cập nhật Model Cards và Datasheet bằng số tổng hợp ở trên. Không ghi model/prediction cấp khách. Kiểm tra nguồn CSV theo SHA-256 đã ghi trong data_sources trước khi ghi tài liệu nghiệm thu; nếu dùng bản xlsx hoặc tập khác, kết quả có thể khác, phải review và không ghi đè tài liệu tham chiếu. So sánh metric chính và threshold với W10/W11, nếu khác phải điều tra môi trường, dữ liệu hoặc code; không chỉnh theo test. Các điều khoản/robots chưa xác minh và xác nhận chính sách lưu giữ của thành viên vẫn là việc nghiệm thu thủ công. Không tuyên bố đã ẩn danh hoàn toàn hoặc đã đạt consent.

In [5]:
import hashlib
csv = ROOT/'data/raw/online_retail_II.csv'
expected_sha = '32569a66f3842a82b0d8c4d63b263c5d98a76bde5d1f65c6c01bf457e541d3a9'
if csv.exists():
    with csv.open('rb') as fh:
        digest = hashlib.file_digest(fh, 'sha256').hexdigest()
else:
    digest = None
same_source = digest == expected_sha
same_results = (len(y_test)==5225 and round(threshold,4)==0.4805
    and np.isclose(clf_table.loc['Overall','roc_auc'], .7936, atol=.0001)
    and np.isclose(clf_table.loc['Overall','accuracy'], .7301, atol=.0001)
    and np.isclose(reg_table.loc['Overall','mae_gbp'],515.34,atol=.01)
    and np.all(pred==0))
print('CSV fingerprint matches W10 reference:', same_source, '| Reference metrics match:', same_results)
if same_source and same_results:
    update_card(ROOT/'docs/model_card_classification.md', clf_table,
        ['n','accuracy','recall','fpr','precision','roc_auc'],
        f'W12 temporal test (đúng CSV tham chiếu), threshold validation {threshold:.12g}; gap recall Intl − UK {signed_gap(clf_table,"recall"):+.4f}.')
    update_card(ROOT/'docs/model_card_regression.md', reg_table,
        ['n','mae_gbp','rmse_gbp','r2','mean_error_gbp'],
        f'W12 temporal test (đúng CSV tham chiếu); gap MAE Intl − UK {signed_gap(reg_table,"mae_gbp"):+.2f} GBP.')
    update_card(ROOT/'docs/datasheet.md', privacy,
        ['customers','suppressed','k_min','customers_below_k','k_target'],
        'W12: mô phỏng generalize/suppress trên test; không công bố bảng khách và không khẳng định dữ liệu gốc ẩn danh.')
    print('Ba tài liệu đã cập nhật aggregate; nhóm cần review trước khi commit.')
else:
    print('Chưa ghi tài liệu tham chiếu: kiểm tra nguồn/metric. Kết quả hiện tại vẫn có trong notebook và CSV cục bộ.')


CSV fingerprint matches W10 reference: True | Reference metrics match: True
Ba tài liệu đã cập nhật aggregate; nhóm cần review trước khi commit.


## 5. Kết luận và checklist Ethics (W12 slide 53)
- Nguồn UCI/CC BY 4.0 và điều khoản Kaggle có trong datasheet; kiểm tra robots và license mirror thủ công nếu chưa xác minh. Không scrape trong pipeline.
- Không công bố mã ID hoặc bảng khách; đo k thực nghiệm, có suppression; lịch sử Git và dữ liệu nguồn vẫn có rủi ro. Không tuyên bố dữ liệu đã vô danh hoàn toàn.
- Định lượng missing ID và độ đại diện theo snapshot; UK có thể chi phối kết quả chung. Dựa vào support và gap thực tế để diễn giải, không khẳng định tác động nhân quả hoặc fairness đã đạt.
- Kết quả Classification/Regression chia nhóm; không chọn lại model dựa trên test. Không triển khai Dummy median; dùng W11 thận trọng trong nghiên cứu hồi cứu.
- AI usage ghi trong README; nhóm phải đọc, kiểm thử, giải thích từng dòng. Review và nộp model card một trang/mô hình cùng báo cáo (Markdown, cần kiểm tra trang khi dàn bản in). Chính sách lưu giữ là đề xuất cần xác nhận thực thi. Không có yêu cầu Spark cho quy mô này.